[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%A9%9F%E7%8E%87%E7%B5%B1%E8%A8%88/07_QQ%E5%9C%96%E5%85%A5%E9%96%80.ipynb) · [回章節首頁](./README.md)

## 這個觀念要幫我們回答什麼？

**核心問題：平均數、標準差都符合目標，就足以說資料是常態嗎？**

- **用在流程哪裡**：模型假設檢查前（例如 GaussianNB 需要特徵在類別內近似常態）。
- **讀完要能做到**：會讀理論分位數 vs 樣本分位數，分得清位置、尺度、形狀三件事。
- **閱讀方式**：每個程式格前先讀「為什麼做／怎麼看」，再執行；用實際輸出回答「先想一想」。

觀念連結：[特徵工程](../名詞解釋/04_特徵工程/README.md)、[常見算法](../名詞解釋/07_常見算法/README.md)

# 07｜Q-Q 圖入門：看形狀，不只看平均

Q-Q 圖是「**排序後的資料 vs 理論分位數**」的散佈圖，用來問：形狀像不像某個參照分布？

**一、先懂分位數：排序後找位置**

第 $p$ 分位數是「有 $p$ 比例資料小於等於它」的門檻值：中位數是 50% 分位數，Q1/Q3 是 25%/75%。注意它是**門檻**，不是區間平均；小樣本時插值定義會影響結果。

**二、Q-Q 圖三要素怎麼讀**

- **橫軸**：理論分位數（例如標準常態的 −2, −1, 0, 1, 2）；**縱軸**：排序後的實際資料。每个点都是一個「同名次配對」。
- **沿著一條直線＝形狀相符**：不必是 $y=x$！$N(170, 8^2)$ 對標準常態作圖仍是直線，只是斜率約 8（**尺度**）、截距約 170（**位置**）。斜率 $\neq 1$ 不代表非常態，它只是在說單位不同。
- **彎曲＝形狀不合**：上彎／下彎、S 形代表偏態或厚尾（本篇對數常態的長右尾會讓右端上飄）。但**小樣本尾端本來就晃**，別拿一兩個點當定論。

**三、兩個關鍵釐清**

1. **標準化修不掉形狀**：常態與對數常態各自標準化後，中心尺度都相同（均值 0、標準差 1），Q-Q 圖仍一眼分辨——因為標準化只動位置尺度，不動形狀（呼應 05）。
2. **Q-Q 圖的點 $\neq$ 散佈圖的點**：散佈圖一點是一筆觀測的 $(x, y)$；Q-Q 圖一點是「理論第 $i$ 名 vs 實際第 $i$ 名」，橫軸不是原始特徵。

SciPy `probplot` 有自己的繪圖位置公式；重點永遠是「選定參照後比較形狀」，貼線不是分布相同的證明，更不是模型準確度。

## 準備共用繪圖設定

**為什麼做**：統一繁體中文字型與配色，讓讀圖重點清楚。

**怎麼看**：這格只載入工具，沒有訓練模型；會列出實際字型。需要套件時由老師協助準備 NumPy、pandas、SciPy、Matplotlib、Seaborn。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

# 設定中文字型支援
# --- 設定 Matplotlib 中文字型支援（Mac / Windows / Colab 通用）---
import glob
import os
try:
    import mplfonts
except ImportError:  # Colab 環境若缺套件則自動安裝（本機已在 requirements.txt）
    %pip install -q mplfonts
    import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)   # 直接註冊 mplfonts 內附的字型，不改任何檔案
use_font("Noto Sans CJK SC")              # 套用思源黑體，負號也會一起處理

sns.set_theme(style='whitegrid', rc={
    'axes.unicode_minus': False, 'figure.facecolor': '#F5F7F2',
    'axes.facecolor': '#FFFFFF', 'axes.spines.top': False,
    'axes.spines.right': False, 'grid.alpha': 0.22,
    'axes.titlesize': 14, 'axes.labelsize': 11, 'figure.dpi': 110})
GREEN, ORANGE, PURPLE = '#2F6657', '#C17B45', '#786497'
rng = np.random.default_rng(42)
print('中文字型：Noto Sans CJK SC（mplfonts 通用寫法）')

## 01｜分位數是在排序後找位置

**為什麼做**：用一小組配送時間找 25%、50%、75% 的數值，為 Q-Q 圖建立直覺。

**怎麼看**：分位數是門檻值，不是每個區間的平均。樣本分位數的精確定義與插值方式可影響小樣本結果。

In [ ]:
times = np.array([10, 12, 13, 15, 18, 20, 25, 32])
positions = [.25, .5, .75]
display(pd.DataFrame({'位置': ['25%', '50%：中位數', '75%'],
                      '對應時間': np.quantile(times, positions)}))

## 02｜同樣常態形狀，可以有不同斜率

**為什麼做**：比較標準常態與 N(170,8²) 對標準常態理論分位數的圖。

**怎麼看**：橫軸是理論分位數，縱軸是排序後的資料。沿一條直線表示形狀較相符，不必是 y=x；斜率與截距反映尺度、位置。

In [ ]:
standard = np.random.default_rng(42).normal(0, 1, size=500)
shifted = 170 + 8 * standard
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title in zip(axes, [standard, shifted], ['標準常態資料', '換成身高尺度：仍是直線']):
    (theoretical, ordered), (slope, intercept, r) = stats.probplot(data, dist='norm')
    ax.scatter(theoretical, ordered, s=10, color=GREEN, alpha=.45)
    ax.plot(theoretical, slope * theoretical + intercept, color=ORANGE, label='擬合參考線')
    ax.set(title=title, xlabel='標準常態理論分位數', ylabel='樣本分位數')
    ax.legend()
    print(f'{title}：參考線斜率 {slope:.2f}、截距 {intercept:.2f}')
plt.show()

## 03｜平均與標準差相同，形狀仍可不同

**為什麼做**：將常態與對數常態樣本各自標準化，讓中心與尺度相同，再比較形狀。

**怎麼看**：兩組平均都約 0、標準差都約 1；對數常態長右尾仍讓點偏離參考線。小樣本尾端自然有波動，別把一個偏離點當定論。

In [ ]:
normal = np.random.default_rng(42).normal(size=800)
skewed = np.random.default_rng(43).lognormal(0, .8, size=800)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, values, title in zip(axes, [normal, skewed], ['常態樣本標準化', '右偏樣本標準化']):
    z = (values - values.mean()) / values.std()
    (theoretical, ordered), (slope, intercept, r) = stats.probplot(z, dist='norm')
    ax.scatter(theoretical, ordered, color=GREEN, alpha=.4, s=10)
    ax.plot(theoretical, slope * theoretical + intercept, color=ORANGE)
    ax.set(title=title, xlabel='常態理論分位數', ylabel='樣本分位數')
plt.show()

## 接回機器學習

這裡用 SciPy probplot 的理論機率圖做常態 Q-Q 入門：它有自己的繪圖位置公式。重點是選定參照分布後比較分位數形狀；貼線不是分布相同的證明，也不是模型準確度。

## 換你解讀

1. 身高 Q-Q 圖的斜率不是 1，就一定不常態嗎？
2. 為什麼標準化無法修掉右偏？
3. Q-Q 圖與散佈圖的每個點代表同一件事嗎？

### 請 AI 幫你做一個小實驗

請 AI 將樣本數從 800 改成 30，重做 Q-Q 圖。先預測尾端是否更不穩定，再比較，不能以一次結果宣稱通用結論。

先說出預期，再看實際圖形，最後寫下「證據 → 解釋 → 不能下的結論」。

[回章節首頁](./README.md)